# Diagnosing the VQE Prototype: H₂ under stretching, optimizers and noise
**IQ Hackathon 2026 · VQE track · Rising Star**

Question: where does the provided VQE prototype lose accuracy, and how much does each fix recover?

## 0. Setup
Installs every package needed (submission rule). The `vqe_lab` helpers install from this project's GitHub repo, which has to be public for this cell to work in Colab.

In [ ]:
%pip install -q numpy scipy matplotlib pylatexenc "qiskit>=2.0" qiskit-aer "qiskit-nature>=0.7" qiskit-ibm-runtime pyscf
%pip install -q git+https://github.com/S2800-0/iq-hackathon-vqe

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import qiskit, qiskit_nature, pyscf
from vqe_lab.hamiltonians import h2, build_molecule, exact_energy
from vqe_lab.vqe import prototype_ansatz, efficient_ansatz, run_vqe, energy, OPTIMIZERS
print("qiskit", qiskit.__version__, "| qiskit-nature", qiskit_nature.__version__, "| pyscf", pyscf.__version__)
CHEM_ACC_MHA = 1.6
SEEDS = [0, 1, 2]

## 1. Baseline: reproduce the prototype at 0.735 Å
Should match the training notebook: −1.857275 Ha (electronic energy).

In [ ]:
mol = h2(0.735)
base = run_vqe(mol.operator, prototype_ansatz(), initial_point=[0.0])
print(f"VQE  {base.energy:.6f} Ha  ({base.nfev} evaluations)")
print(f"exact {exact_energy(mol.operator):.6f} Ha   HF {mol.hf_energy:.6f} Ha")

## 2. Energy curve: exact and Hartree–Fock references

In [ ]:
bonds = np.linspace(0.5, 2.5, 11)
mols = [h2(d) for d in bonds]
exact = np.array([exact_energy(m.operator) for m in mols])
hf = np.array([m.hf_energy for m in mols])
nuc = np.array([m.nuclear_repulsion for m in mols])

fig, ax = plt.subplots()
ax.plot(bonds, exact + nuc, "k-", label="exact")
ax.plot(bonds, hf + nuc, "--", label="Hartree–Fock")
ax.set(xlabel="H–H bond length (Å)", ylabel="total energy (Ha)", title="H₂ dissociation curve")
ax.legend(); plt.show()

## 3. Ansatz comparison (noiseless)
Finding from testing: on 2-qubit H₂ the 1-parameter prototype is already exact at every bond length. Without noise, the ansatz isn't what limits accuracy here. TODO: write up why (symmetry: one parameter spans the whole relevant subspace).

In [ ]:
proto = [run_vqe(m.operator, prototype_ansatz(), initial_point=[0.0]) for m in mols]
eff = [run_vqe(m.operator, efficient_ansatz(2, reps=2), seed=0, maxiter=500) for m in mols]

fig, ax = plt.subplots()
ax.semilogy(bonds, [max((r.energy - e) * 1e3, 1e-7) for r, e in zip(proto, exact)], "o-", label="prototype (1 param)")
ax.semilogy(bonds, [max((r.energy - e) * 1e3, 1e-7) for r, e in zip(eff, exact)], "s-", label="efficient_su2 (reps=2)")
ax.axhline(CHEM_ACC_MHA, color="gray", ls=":", label="chemical accuracy")
ax.set(xlabel="H–H bond length (Å)", ylabel="error vs exact (mHa)", title="Ansatz error, noiseless")
ax.legend(); plt.show()

## 4. Optimizer comparison
TODO: for 2–3 bond lengths and each optimizer: final error (mHa) and number of evaluations, over SEEDS. Add SPSA once it's implemented.

In [ ]:
# TODO: loop over OPTIMIZERS x [0.735, 1.5, 2.5] x SEEDS, collect error and nfev, bar chart

## 5. Warm start along the curve
TODO: pass each geometry's best params to the next; compare total evaluations cold vs warm, mean and range over SEEDS.

In [ ]:
# TODO

## 6. Noise and readout mitigation
Same circuit, same optimal angle, three ways of measuring it:
- **ideal**: real shots, no hardware noise (shot noise only)
- **noisy**: Aer simulator built from IBM's `FakeManilaV2` calibration (gate + readout errors)
- **mitigated**: noisy, plus readout-error mitigation (inverse of measured per-qubit assignment matrices)

The angle is the noiseless VQE optimum, so this isolates the measurement error from the optimizer.

In [ ]:
from vqe_lab.noise import SamplingEstimator

NOISE_BONDS = [0.735, 1.5, 2.5]
SHOTS = 8000
settings = {"ideal": dict(noisy=False), "noisy": dict(noisy=True),
            "mitigated": dict(noisy=True, mitigate_readout=True)}
noise_results = {k: {} for k in settings}
for d in NOISE_BONDS:
    m = h2(d)
    theta = run_vqe(m.operator, prototype_ansatz(), initial_point=[0.0]).params
    ex = exact_energy(m.operator)
    for name, kw in settings.items():
        errs = [(energy(m.operator, prototype_ansatz(), theta,
                        SamplingEstimator(shots=SHOTS, seed=s, **kw)) - ex) * 1e3 for s in SEEDS]
        noise_results[name][d] = {"mean": float(np.mean(errs)), "min": min(errs), "max": max(errs)}

x = np.arange(len(NOISE_BONDS)); w = 0.27
fig, ax = plt.subplots()
for i, name in enumerate(settings):
    r = noise_results[name]
    mean = np.array([r[d]["mean"] for d in NOISE_BONDS])
    lo = mean - np.array([r[d]["min"] for d in NOISE_BONDS])
    hi = np.array([r[d]["max"] for d in NOISE_BONDS]) - mean
    ax.bar(x + (i - 1) * w, mean, w, yerr=[lo, hi], capsize=3, label=name)
ax.axhline(CHEM_ACC_MHA, color="gray", ls=":", label="chemical accuracy")
ax.set_xticks(x, [f"{d} Å" for d in NOISE_BONDS])
ax.set(xlabel="H–H bond length", ylabel="energy error vs exact (mHa)",
       title=f"Noise and readout mitigation (FakeManilaV2, {SHOTS} shots, {len(SEEDS)} seeds)")
ax.legend(); plt.show()

for name in settings:
    print(name.ljust(10), "  ".join(f"{d} Å: {noise_results[name][d]['mean']:6.2f} mHa" for d in NOISE_BONDS))

## 7. Error budget
TODO: stacked bars at one geometry: ansatz | optimizer | shots | hardware noise.

In [ ]:
# TODO

## 8. (Optional) HeH⁺ and real IBM hardware

In [ ]:
# TODO

## 9. Summary table
TODO: every number quoted in the report, printed here.